In [0]:
import requests
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

import pandas as pd

In [0]:
CATALOG = "sap_lakehouse"

In [0]:
import uuid
from datetime import datetime

RUN_ID = str(uuid.uuid4())

PIPELINE_NAME = "sap_business_partner_pipeline"

PIPELINE_START_TIME = datetime.now()

In [0]:
BRONZE_BP_TABLE = f"{CATALOG}.bronze.business_partner_raw"
SILVER_BP_TABLE = f"{CATALOG}.silver.business_partner_current"
GOLD_BP_TABLE = f"{CATALOG}.gold.business_partner_summary"
AUDIT_TABLE = f"{CATALOG}.audit.business_partner_pipeline_runs"
WATERMARK_TABLE = f"{CATALOG}.audit.business_partner_watermark"

In [0]:
SAP_API_KEY = "OfL8fCMhGICU0F7J5HUl2tPRdqnS1T7R"

url = "https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata/sap/API_BUSINESS_PARTNER/A_BusinessPartner?$top=100"

headers = {
    "APIKey": SAP_API_KEY,
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)

print(response.status_code)

In [0]:
sap_data = response.json()

results = sap_data["d"]["results"]

print(f"Records pulled: {len(results)}")

In [0]:
pandas_df = pd.DataFrame(results)

bp_df = spark.createDataFrame(pandas_df)

display(bp_df)

In [0]:
bronze_df = (
    bp_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("BusinessPartner"),
                F.col("BusinessPartnerFullName"),
                F.col("BusinessPartnerCategory")
            ),
            256
        )
    )
)

In [0]:
clean_bp_df = bp_df.select(
    "BusinessPartner",
    "BusinessPartnerCategory",
    "BusinessPartnerFullName",
    "BusinessPartnerName",
    "OrganizationBPName1",
    "OrganizationBPName2",
    "CreatedByUser",
    "CreationDate",
    "CreationTime",
    "BusinessPartnerUUID",
    "CorrespondenceLanguage",
    "FirstName",
    "LastName",
    "IsFemale",
    "IsMale",
    "Customer",
    "Supplier"
)

In [0]:
clean_bp_df = clean_bp_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

In [0]:
clean_bp_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(BRONZE_BP_TABLE)

In [0]:
bronze_count = spark.table(BRONZE_BP_TABLE).count()

In [0]:
display(
    spark.table(BRONZE_BP_TABLE)
)

**Silver Layer Data Preparation**

Deduplicate Business Partners

In [0]:
bp_window = Window.partitionBy("BusinessPartnerUUID").orderBy(F.col("ingestion_timestamp").desc())

Keep Latest Record

In [0]:
latest_bp_df = (
    bronze_df
    .withColumn(
        "row_num",
        F.row_number().over(bp_window)

    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

Add Processing Timestamp

In [0]:
latest_bp_df = latest_bp_df.withColumn(
    "silver_processed_at",
    F.current_timestamp()
)

In [0]:
silver_clean_df = latest_bp_df.select(
    F.col("BusinessPartner").cast("string"),
    F.col("BusinessPartnerCategory").cast("string"),
    F.col("BusinessPartnerFullName").cast("string"),
    F.col("BusinessPartnerName").cast("string"),
    F.col("OrganizationBPName1").cast("string"),
    F.col("OrganizationBPName2").cast("string"),
    F.col("BusinessPartnerUUID").cast("string"),
    F.col("CorrespondenceLanguage").cast("string"),
    F.col("CreatedByUser").cast("string"),
    F.col("CreationDate").cast("string"),
    F.col("Customer").cast("string"),
    F.col("Supplier").cast("string"),
    F.col("ingestion_timestamp"),
    F.col("silver_processed_at")
)

Write Silver Table

In [0]:
silver_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_BP_TABLE)

In [0]:
silver_count = spark.table(SILVER_BP_TABLE).count()

In [0]:
display(
    spark.table(SILVER_BP_TABLE)
    )


In [0]:
#GOLD_BP_TABLE = f"{CATALOG}.gold.business_partner_summary"

In [0]:
gold_bp_df = (
    spark.table(SILVER_BP_TABLE)
    .groupBy(
        "BusinessPartnerCategory",
        "CorrespondenceLanguage"
    )
    .agg(
        F.count("*").alias("total_business_partners")
    )
    .withColumn(
        "gold_processed_at",
        F.current_timestamp()
    )
)

In [0]:
gold_bp_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_BP_TABLE)

In [0]:
gold_count = spark.table(GOLD_BP_TABLE).count()

In [0]:
display(
    spark.table(GOLD_BP_TABLE)
)

In [0]:
from pyspark.sql.types import *

watermark_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("last_successful_run", TimestampType(), True)
])

initial_watermark_df = spark.createDataFrame(
    [
        ("sap_business_partner_pipeline", None)
    ],
    schema=watermark_schema
)

initial_watermark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(WATERMARK_TABLE)

In [0]:
display(
    spark.table(WATERMARK_TABLE)
)

In [0]:
audit_schema = StructType([

    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("pipeline_start_time", TimestampType(), True),
    StructField("pipeline_end_time", TimestampType(), True),

    StructField("records_pulled_from_api", IntegerType(), True),
    StructField("records_written_to_bronze", IntegerType(), True),
    StructField("records_written_to_silver", IntegerType(), True),
    StructField("records_written_to_gold", IntegerType(), True),

    StructField("duplicates_removed", IntegerType(), True),
    StructField("quarantined_records", IntegerType(), True),

    StructField("pipeline_status", StringType(), True),
    StructField("error_message", StringType(), True),

    StructField("audit_created_at", TimestampType(), True)

])

In [0]:
empty_audit_df = spark.createDataFrame([], audit_schema)

empty_audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(AUDIT_TABLE)

In [0]:
PIPELINE_END_TIME = datetime.now()

In [0]:
valid_bp_df = bp_df.filter(
    F.col("BusinessPartner").isNotNull()
)

In [0]:
bad_bp_df = bp_df.filter(
    F.col("BusinessPartner").isNull()
)

In [0]:
records_pulled = bp_df.count()

duplicates_removed_count = (
    valid_bp_df.count() - latest_bp_df.count()
)

quarantined_count = bad_bp_df.count()

In [0]:
audit_data = [

    (
        RUN_ID,
        PIPELINE_NAME,
        PIPELINE_START_TIME,
        PIPELINE_END_TIME,

        records_pulled,
        bronze_count,
        silver_count,
        gold_count,

        duplicates_removed_count,
        quarantined_count,

        "SUCCESS",
        None,

        datetime.now()
    )

]

audit_df = spark.createDataFrame(
    audit_data,
    schema=audit_schema
)

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(AUDIT_TABLE)

In [0]:
display(
    spark.table(AUDIT_TABLE)
)